**!!! NOTEBOOK FOR USAGE REFERENCE !!!**

This notebook is not intended to provide actual code; Instead, it is just a "getting started", something you can take as inspiration when using this testbed.

For developers: please do not commit the outputs to the library. Always clear all outputs before saving.

----

Downloads, enrich, filter, save splits

Generates metadata_filtered

General notebook structure:
1. Inicial set: ~300 entities to be partially enriched
2. Enriched set: fully enriched and verified, including manual checks/inputs. Manually remove mismaches name<->pantheon.
3. Filtered set: filtered 100 entities based on representativenss balancing
4. Entities appear exactly in this order in the table representations.
5. Images should only be saved for the entities in the final enriched set


In [ ]:
import os
import sys
import json
import math
from typing import List, Dict
import dotenv
import pandas as pd
import scipy.io as sio
import torch.utils.checkpoint

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 30)

sys.path.append('../TRDP-unlearning')
dotenv.load_dotenv('../TRDP-unlearning/SD_lora_distil/.env')
os.environ['WANDB_DISABLED'] = "true"
assert len(os.getenv('HF_TOKEN'))>0
#!huggingface-cli login --token ${HF_TOKEN}

from vision_unlearning.datasets import split_dataset_sun, balanced_subsample_lib
from vision_unlearning.datasets.testbed import calculate_similarity_clip, plot_heatmap
from vision_unlearning.utils.logger import get_logger, setup_loggers


logger = get_logger('testbed')
setup_loggers()

In [ ]:
dataset_base_path = 'assets/datasets/SUN'
dataset_base_path_filtered = 'assets/datasets/SUN_splits_filtered'

#!rm "assets/similarity_clip_scenes.json"
#!rm -rf {dataset_base_path}
#!rm assets/metadata_scenes_1_enriched_but_not_filtered.json && rm assets/metadata_scenes_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
#!rm assets/metadata_scenes_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
#!rm -r {dataset_base_path_filtered}

In [ ]:
##############################
# Step 1: prepare datasets
##############################
hierarchy: Dict[str, List[str]] = {
    "Functions/Affordances": [
        "camping",
        "hiking",
        "shopping",
        "farming",
        "studying/ learning",
        "diving",
        "praying",
        "climbing",
        "research",
        "swimming",
        "sports",
        "spectating/ being in an audience",
        "reading",
        "sailing/ boating",
        "driving",
        "conducting business",
        "sunbathing",
        "socializing",
        "eating",
        "waiting in line/ queuing",
        "teaching/ training",
        "working",
        "biking",
        "playing",
        "competing",
        "congregating",
        "transporting things or people",
        "using tools",
        "exercise",
        "vacationing/ touring"
    ],
    "Materials": [
        "foliage",
        "vegetation",
        "shrubbery",
        "trees",
        "leaves",
        "clouds",
        "snow",
        "grass",
        "asphalt",
        "sand",
        "running water",
        "dirt/soil",
        "ocean",
        "shingles",
        "paper",
        "brick",
        "flowers",
        "rock/stone",
        "carpet",
        "still water",
        "rubber/ plastic",
        "wood (not part of a tree)",
        "pavement",
        "cloth",
        "vinyl/ linoleum",
        "glass",
        "concrete",
        "fencing",
        "wire",
        "metal"
    ]
,
    "Surface Properties": [
        "natural light",
        "direct sun/sunny",
        "electric/indoor lighting",
        "dirty",
        "moist/ damp",
        "aged/ worn",
        "glossy",
        "rusty",
        "matte",
        "dry"
    ],
    "Spatial Envelope": [
        "natural",
        "far-away horizon",
        "enclosed area",
        "open area",
        "rugged scene",
        "cold",
        "no horizon",
        "cluttered space",
        "mostly vertical components",
        "stressful",
        "man-made",
        "symmetrical",
        "soothing",
        "warm",
        "semi-enclosed area",
        "mostly horizontal components",
    ]
}

#!rm -rf {dataset_base_path}
if not os.path.exists(os.path.join(dataset_base_path, "SUNAttributeDB")) or not os.path.exists(os.path.join(dataset_base_path, "images")):
    os.makedirs(dataset_base_path, exist_ok=True)
    !wget https://cs.brown.edu/~gmpatter/Attributes/SUNAttributeDB.tar.gz -O {os.path.join(dataset_base_path, "SUNAttributeDB.tar.gz")}
    !wget https://cs.brown.edu/~gmpatter/Attributes/SUNAttributeDB_Images.tar.gz -O {os.path.join(dataset_base_path, "SUNAttributeDB_Images.tar.gz")}
    !tar -xzf {os.path.join(dataset_base_path, "SUNAttributeDB.tar.gz")} -C {dataset_base_path}
    !tar -xzf {os.path.join(dataset_base_path, "SUNAttributeDB_Images.tar.gz")} -C {dataset_base_path}

attributes: List[str] = [f[0][0] for f in sio.loadmat(os.path.join(dataset_base_path, "SUNAttributeDB/attributes.mat"))['attributes'].tolist()]
filenames: List[str] = [f[0][0] for f in sio.loadmat(os.path.join(dataset_base_path, "SUNAttributeDB/images.mat"))['images'].tolist()]

df = pd.DataFrame(
    sio.loadmat(os.path.join(dataset_base_path, "SUNAttributeDB/attributeLabels_continuous.mat"))['labels_cv'],
    index=filenames,
    columns=attributes,
)
df = df>0.5  # Convert continuous to binary; 2 out of 3 labelers agree with the label
df['category'] = pd.Series(df.index, index=df.index).str.split('/').apply(lambda parts: '_'.join(parts[1:-1]))
print(df.shape)

# assert sum([len(hierarchy[k]) for k in hierarchy]) == df.shape[1]-1, 'Not all attributes are mapped in the hierarchy'
assert all([all([att in df.columns for att in hierarchy[k]]) for k in hierarchy]), 'There are attributes that dont exist'

In [ ]:
##############################
# Step 2: attribute inference
##############################
#!rm assets/metadata_scenes_1_enriched_but_not_filtered.json && rm assets/metadata_scenes_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
if os.path.exists('assets/metadata_scenes_1_enriched_but_not_filtered.json'):
    with open(f"assets/metadata_scenes_1_enriched_but_not_filtered.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    # Attribute inference option 1:labels from SUN itself
    # Not present

    # Attribute inference option 2: from another dataset
    # Not found. TODO

    # Attribute inference option 3: average value in SUN dataset
    df_categories = df.groupby('category').mean()
    print((df_categories>0.1).mean().mean()*100)
    print((df_categories>0.5).mean().mean()*100)
    print((df_categories>0.8).mean().mean()*100)
    df_categories = df_categories>0.5   # At least x% of the instances have this property

    df_categories.reset_index(inplace=True)
    df_categories.rename({'category': 'name'}, axis=1, inplace=True)
    df_categories['dataset_n_original'] = 20
    metadata = df_categories.to_dict(orient='records')
    
    # Save
    with open(f"assets/metadata_scenes_1_enriched_but_not_filtered.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

In [ ]:
##############################
# Step 3: filter
##############################
# Some interesting attributes:
#df_categories.value_counts(['socializing', 'natural'])
#df_categories.value_counts(['socializing', 'open area'])
#df_categories.value_counts(['exercise', 'natural'])
#df_categories.value_counts(['sports', 'natural'])
#df_categories.value_counts(['sports', 'open area'])
#df_categories[df_categories['man-made'] & df_categories['ocean']]
#df_categories.value_counts(['driving', 'transporting things or people'])

#!rm assets/metadata_scenes_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
balanced_n = 100

if os.path.exists('assets/metadata_scenes_2_enriched_filtered.json'):
    logger.info('Reading existing filtered data')
    with open(f"assets/metadata_scenes_2_enriched_filtered.json", "r", encoding="utf-8") as f:
        metadata_filtered = json.load(f)
else:
    logger.info('Filtering')
    with open(f"assets/metadata_scenes_1_enriched_but_not_filtered.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    df_enriched = pd.DataFrame(metadata)
    print(f"Rows in unfiltered dataset: {df_enriched.shape[0]}")

    ########################
    # make ideal choice of attributes to balance, constrained to one of "Functions/Affordances" and another from "Spatial Envelope" category groups
    # This sequential optimization is suboptimal... but is probably good enough, and I like the selected attributes
    min_1 = 0
    min_1_att = None
    for att_1 in hierarchy["Functions/Affordances"]:
        min_current = df_enriched.value_counts([att_1]).min()
        if min_current > min_1 and min_current != df_enriched.value_counts([att_1]).max():
            min_1 = min_current
            min_1_att = att_1
    print(f"First selected attributes: {min_1_att}, with min count {min_1}")

    min_2 = 0
    min_2_att = None
    for att_2 in hierarchy["Spatial Envelope"]:
        #if att_2 in ['man-made', 'natural']:
        #    continue#
        if df_enriched[att_2].nunique() == 1:
            continue
        #if len(df_enriched.value_counts([min_1_att, att_2])) < 4:
        #    continue
        min_current = df_enriched.value_counts([min_1_att, att_2]).min()
        if min_current > min_2:
            min_2 = min_current
            min_2_att = att_2
    print(f"Second selected attributes: {min_2_att}, with min count {min_2}")
    ########################

    print('BEFORE BALANCING')
    df_enriched.dropna(subset=[min_1_att, min_2_att], axis=0, inplace=True)
    print(df_enriched.value_counts(min_1_att))
    print("-"*50)
    print(df_enriched.value_counts(min_2_att))
    print("-"*50)
    counts = df_enriched.value_counts([min_1_att, min_2_att])
    print(counts)
    print(f"Rows after dropping rows that could not be enriched: {df_enriched.shape[0]}")
    print(f"Need {math.ceil(100/len(counts))}, have min {counts.min()}")

    df_balanced = balanced_subsample_lib(df_enriched, group_cols=[min_1_att, min_2_att], target=balanced_n, priority_col='dataset_n_original')
    print('-'*50)
    print('-')
    print('-'*50)
    print('BALANCED')

    print(df_balanced.value_counts(min_1_att))
    print('-'*50)
    print(df_balanced.value_counts(min_2_att))
    print('-'*50)
    print(df_balanced.value_counts([min_1_att, min_2_att]))

    metadata_filtered = df_balanced.to_dict(orient='records')
    with open(f"assets/metadata_scenes_2_enriched_filtered.json", "w", encoding="utf-8") as f:
        json.dump(metadata_filtered, f, indent=2)

assert type(metadata_filtered) == list
assert len(metadata_filtered) == balanced_n


In [ ]:
##############################
# Step 4: save splits
##############################
smallest_entity = min([entity['dataset_n_original'] for entity in metadata_filtered])
restrict_labels = [e['name'] for e in metadata_filtered]
logger.info(f"We have {balanced_n} identities, each one with {smallest_entity} images")

#!rm -r {dataset_base_path_filtered}
if not os.path.exists(dataset_base_path_filtered):
    for i, target in enumerate(restrict_labels):
        logger.info(f"Saving split dataset for entity {i}: {target}")
        dataset_forget_name = f"{dataset_base_path_filtered}/{target}/train_forget"
        dataset_retain_name = f"{dataset_base_path_filtered}/{target}/train_retain"
        class_to_number = split_dataset_sun(
            dataset_base_path,
            dataset_forget_name,
            dataset_retain_name,
            target,
            forget_max_img = smallest_entity,
            retain_max_img_per_class= smallest_entity,
            restrict_labels = restrict_labels,
        )
        assert sum([v>0 for v in class_to_number.values()]) == balanced_n, 'More entities than expected were saved'
        assert sum([v==smallest_entity for v in class_to_number.values()]) == balanced_n, 'Not all entities have the same number of images'
        assert target in class_to_number.keys()
    
        assert sum(len(files) for _, _, files in os.walk(dataset_forget_name)) == smallest_entity+1
        assert sum(len(files) for _, _, files in os.walk(dataset_retain_name)) == (balanced_n-1)*smallest_entity+1

        #if i > 5:
        #    break

# Each identity has about 8.2Mb
#!du -hs "{dataset_base_path_filtered}/{target}"

#!find "{dataset_forget_name}" \( -type f -o -type l \) | wc -l
#!find "{dataset_retain_name}"  \( -type f -o -type l \) | wc -l

In [ ]:
##############################
# Step 5: similarity matrix
##############################
# TODO: move to standalone **Notebook 2: Data Exploration**
#!rm "assets/similarity_clip_scenes.json"
df_similarities_clip = calculate_similarity_clip('scenes', restrict_labels)
plot_heatmap(df_similarities_clip)
df_similarities_clip.head()